In [3]:
import pandas as pd
import numpy as np
import os
# Read the CSV files

data_dir = '..\data\specimens_export_20251028_192412'
specimens_path = os.path.join(data_dir, 'specimens.csv')
tensile_path = os.path.join(data_dir, 'tensile_specimens.csv')
polish_path = os.path.join(data_dir, 'polish_specimens.csv')
notch_path = os.path.join(data_dir, 'notch_specimens.csv')

specimens_df = pd.read_csv(specimens_path, sep=';')
tensile_df = pd.read_csv(tensile_path, sep=';')
polish_df = pd.read_csv(polish_path, sep=';')
notch_df = pd.read_csv(notch_path, sep=';')

# Display basic information about each dataframe
print("=== SPECIMENS (Main) ===")
print(f"Shape: {specimens_df.shape}")
print(f"Columns: {list(specimens_df.columns)}")
print("\n=== TENSILE SPECIMENS ===")
print(f"Shape: {tensile_df.shape}")
print(f"Columns: {list(tensile_df.columns)}")
print("\n=== POLISH SPECIMENS ===")
print(f"Shape: {polish_df.shape}")
print(f"Columns: {list(polish_df.columns)}")
print("\n=== NOTCH SPECIMENS ===")
print(f"Shape: {notch_df.shape}")
print(f"Columns: {list(notch_df.columns)}")



# %%
# Function to link all tables together
def create_linked_dataset():
    """
    Link all specimen tables together using the key relationships
    """
    
    # Start with the main specimens table
    linked_data = specimens_df.copy()
    
    # Add tensile test data
    tensile_data = tensile_df.add_suffix('_tensile').rename(columns={'specimens_key_tensile': 'key'})
    linked_data = linked_data.merge(tensile_data, on='key', how='left')
    
    # Add polish test data
    polish_data = polish_df.add_suffix('_polish').rename(columns={'specimens_key_polish': 'key'})
    linked_data = linked_data.merge(polish_data, on='key', how='left')
    
    # Add notch test data
    notch_data = notch_df.add_suffix('_notch').rename(columns={'specimens_key_notch': 'key'})
    linked_data = linked_data.merge(notch_data, on='key', how='left')
    
    return linked_data

# Create the linked dataset
complete_dataset = create_linked_dataset()

print(f"Complete dataset shape: {complete_dataset.shape}")
print(f"Original specimens: {len(specimens_df)}")
print(f"With tensile data: {complete_dataset['name_tensile'].notna().sum()}")
print(f"With polish data: {complete_dataset['name_polish'].notna().sum()}")
print(f"With notch data: {complete_dataset['name_notch'].notna().sum()}")

# Define the image URL columns
image_url_columns = [
    'polish_img_url_3a_polish',
    'polish_img_url_6a_polish', 
    'polish_img_url_1_orig_polish',
    'polish_img_url_2_orig_polish',
    'polish_img_url_3_orig_polish',
    'polish_img_url_4_orig_polish',
    'polish_img_url_5_orig_polish',
    'polish_img_url_6_orig_polish',
    'polish_img_url_3a_orig_polish',
    'polish_img_url_6a_orig_polish'
]

# Filter rows where at least one image URL column is not null
has_images_mask = complete_dataset[image_url_columns].notna().any(axis=1)

# Get rows with context (DataFrame) - only specimens that have at least one image
specimens_with_images = complete_dataset.loc[
    has_images_mask,
    ['key'] + image_url_columns
]

print(f"Found {len(specimens_with_images)} specimens with at least one image URL")

# %%
for key in  complete_dataset.keys():
    print(key)

# %%
# Define the image URL columns
image_url_columns = [
    'polish_img_url_3a_polish',
    'polish_img_url_6a_polish', 
    'polish_img_url_1_orig_polish',
    'polish_img_url_2_orig_polish',
    'polish_img_url_3_orig_polish',
    'polish_img_url_4_orig_polish',
    'polish_img_url_5_orig_polish',
    'polish_img_url_6_orig_polish',
    'polish_img_url_3a_orig_polish',
    'polish_img_url_6a_orig_polish'
]

# Filter rows where at least one image URL column is not null
has_images_mask = complete_dataset[image_url_columns].notna().any(axis=1)

# Get rows with context (DataFrame) - only specimens that have at least one image
specimens_with_images = complete_dataset.loc[
    has_images_mask,
    ['key'] + image_url_columns
]

print(f"Found {len(specimens_with_images)} specimens with at least one image URL")

# %%
# Function to extract just the filename from URL paths
def extract_filename(url_string):
    """
    Extract the last filename from a URL string that may contain multiple paths separated by semicolons
    Example: "/documents/generic-form/.../RE-C015-S9_9bf94606_.bmp;documents/generic-form/.../RE-C015-S9.bmp" 
    Returns: "RE-C015-S9.bmp"
    """
    if pd.isna(url_string) or url_string == '':
        return url_string
    
    # Split by semicolon and take the last part
    parts = str(url_string).split(';')
    last_part = parts[-1]
    
    # Extract just the filename (everything after the last /)
    filename = last_part.split('/')[-1]
    
    return filename

# Apply the function to all image URL columns in specimens_with_images
for col in image_url_columns:
    if col in specimens_with_images.columns:
        specimens_with_images[col] = specimens_with_images[col].apply(extract_filename)

# Display the cleaned data

# %%
specimens_with_images

# %%
import os

# Read all image files from the images directory
images_dir = os.path.join(data_dir, 'images')
if os.path.exists(images_dir):
    # Get all files in the images directory
    image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]
    print(f"Found {len(image_files)} files in images directory")
    
    # Convert to set for faster lookup
    image_files_set = set(image_files)
    
    # Get all filenames from specimens_with_images (after cleaning)
    database_filenames = set()
    for col in image_url_columns:
        if col in specimens_with_images.columns:
            # Get non-null values and add to set
            non_null_values = specimens_with_images[col].dropna()
            database_filenames.update(non_null_values)
    
    print(f"Found {len(database_filenames)} unique filenames in database")
    
    # Find matches
    matches = image_files_set.intersection(database_filenames)
    print(f"Found {len(matches)} matching files")
    
    # Find files in directory but not in database
    files_not_in_db = image_files_set - database_filenames
    print(f"Files in directory but not in database: {len(files_not_in_db)}")
    
    # Find filenames in database but not in directory
    db_not_in_files = database_filenames - image_files_set
    print(f"Database entries not found in directory: {len(db_not_in_files)}")
    
    # Display some examples
    print("\nFirst 10 matches:")
    for i, match in enumerate(sorted(matches)):
        if i < 10:
            print(f"  {match}")
        else:
            break
    
    print("\nFirst 10 files not in database:")
    for i, file in enumerate(sorted(files_not_in_db)):
        if i < 10:
            print(f"  {file}")
        else:
            break
    
    print("\nFirst 10 database entries not found:")
    for i, db_file in enumerate(sorted(db_not_in_files)):
        if i < 10:
            print(f"  {db_file}")
        else:
            break
    
else:
    print(f"Images directory not found: {images_dir}")

# %% [markdown]
# complete_dataset['complete_dataset']

# %%
# Let's examine the data distribution and relationships
print("=== DATA SUMMARY ===")

# Count specimens by material
print("\nSpecimens by Material:")
print(specimens_df['material'].value_counts())

# Count specimens by customer
print("\nTop 10 Customers:")
print(specimens_df['customer_name'].value_counts().head(10))

# Check tensile test results
print("\nTensile Test Summary:")
if not tensile_df.empty:
    numeric_cols = tensile_df.select_dtypes(include=[np.number]).columns
    print(f"Available numeric columns: {list(numeric_cols)}")
    if len(numeric_cols) > 0:
        print(tensile_df[numeric_cols].describe())

# Check polish test results
print("\nPolish Test Summary:")
polish_with_data = polish_df[polish_df['porosity'].notna()]
if not polish_with_data.empty:
    print(f"Specimens with porosity data: {len(polish_with_data)}")
    print(f"Porosity range: {polish_with_data['porosity'].min():.2f} - {polish_with_data['porosity'].max():.2f}")
    print(f"Average porosity: {polish_with_data['porosity'].mean():.2f}")

# %%
# Create some useful views of the linked data
def create_analysis_views(complete_df):
    """
    Create different views of the data for analysis
    """
    
    # View 1: Specimens with tensile test results
    tensile_view = complete_df[complete_df['name_tensile'].notna()].copy()
    
    # View 2: Specimens with polish/metallography results
    polish_view = complete_df[complete_df['name_polish'].notna()].copy()
    
    # View 3: Specimens with notch test results
    notch_view = complete_df[complete_df['name_notch'].notna()].copy()
    
    # View 4: Complete specimens (with all test types)
    complete_tests = complete_df[
        (complete_df['name_tensile'].notna()) & 
        (complete_df['name_polish'].notna()) & 
        (complete_df['name_notch'].notna())
    ].copy()
    
    return {
        'tensile_view': tensile_view,
        'polish_view': polish_view,
        'notch_view': notch_view,
        'complete_tests': complete_tests,
        'full_dataset': complete_df
    }

# Create analysis views
analysis_views = create_analysis_views(complete_dataset)

print("=== ANALYSIS VIEWS ===")
for view_name, view_df in analysis_views.items():
    print(f"{view_name}: {len(view_df)} specimens")

# %%
analysis_views['polish_view']

=== SPECIMENS (Main) ===
Shape: (2437, 26)
Columns: ['key', 'name', 'description', 'number', 'sub_sales_order_key', 'sub_sales_order_number', 'customer_name', 'material', 'build_job_key', 'build_job_number', 'print_sub_sales_order_key', 'machine', 'powder_id', 'lab_run_number', 'related_files', 'comment', 'polish_comment', 'tensile_comment', 'notch_comment', 'chemical_comment', 'rigid_comment', 'fatigue_comment', 'external_comment', 'flag_import', 'no_specimens_detail', 'no_specimens_total']

=== TENSILE SPECIMENS ===
Shape: (1623, 16)
Columns: ['key', 'name', 'description', 'number', 'specimens_key', 'norm', 'shape_of_specimens', 'orientation', 'temperature', 'yield_limit', 'yield_strength', 'tensile_strength', 'elongation_at_break', 'quadrant_build_in', 'heat_treatment', 'contraction_at_break']

=== POLISH SPECIMENS ===
Shape: (7093, 66)
Columns: ['key', 'name', 'description', 'number', 'specimens_key', 'density', 'volume_energy_density', 'quadrant_build_in', 'heat_treatment', 'polis

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_30732\3835228897.py:6: SyntaxWarning: invalid escape sequence '\d'
  data_dir = '..\data\specimens_export_20251028_192412'


,key,name,description,number,sub_sales_order_key,sub_sales_order_number,customer_name,material,build_job_key,build_job_number,...,porosity_3a_polish,grain_size_6a_polish,key_notch,name_notch,description_notch,number_notch,shape_notch,notch_bending_work_notch,quadrant_build_in_notch,heat_treatment_notch
17,2d2ac769c4b4aabd7f35fabd34e45b6d,RE-A725,NaN,RE-A725,NaN,21-3958-1,Rosswag GmbH Engineering,1.4404,NaN,01791,...,NaN,NaN,058978c625fa10c372756e177408319c,NaN,NaN,K001,Charpy V,0.0,NaN,as-built
18,2d2ac769c4b4aabd7f35fabd34e45b6d,RE-A725,NaN,RE-A725,NaN,21-3958-1,Rosswag GmbH Engineering,1.4404,NaN,01791,...,NaN,NaN,058978c625fa10c372756e177408319c,NaN,NaN,K001,Charpy V,0.0,NaN,as-built
19,2d2ac769c4b4aabd7f35fabd34e45b6d,RE-A725,NaN,RE-A725,NaN,21-3958-1,Rosswag GmbH Engineering,1.4404,NaN,01791,...,NaN,NaN,058978c625fa10c372756e177408319c,NaN,NaN,K001,Charpy V,0.0,NaN,as-built
20,2d2ac769c4b4aabd7f35fabd34e45b6d,RE-A725,NaN,RE-A725,NaN,21-3958-1,Rosswag GmbH Engineering,1.4404,NaN,01791,...,NaN,NaN,058978c625fa10c372756e177408319c,NaN,NaN,K001,Charpy V,0.0,NaN,as-built
21,2d2ac769c4b4aabd7f35fabd34e45b6d,RE-A725,NaN,RE-A725,NaN,21-3958-1,Rosswag GmbH Engineering,1.4404,NaN,01791,...,NaN,NaN,058978c625fa10c372756e177408319c,NaN,NaN,K001,Charpy V,0.0,NaN,as-built
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22709,5e5743e9df98d3fa9c5055571623373f,RE-A447,NaN,RE-A447,NaN,NaN,Rosswag GmbH,Ni-1486-L75,NaN,01531,...,NaN,NaN,9df74ac3da55b295a7425df104bc6d50,RE-A447,NaN,K003,NaN,0.0,NaN,as-built
22710,5e5743e9df98d3fa9c5055571623373f,RE-A447,NaN,RE-A447,NaN,NaN,Rosswag GmbH,Ni-1486-L75,NaN,01531,...,NaN,NaN,d2430601aea421c87f4e793f45ea7fab,RE-A447,NaN,K002,NaN,0.0,NaN,as-built
22711,5e5743e9df98d3fa9c5055571623373f,RE-A447,NaN,RE-A447,NaN,NaN,Rosswag GmbH,Ni-1486-L75,NaN,01531,...,NaN,NaN,815fafac0776abf65eb391cb6e3140be,RE-A447,NaN,K001,NaN,0.0,NaN,as-built
22712,5e5743e9df98d3fa9c5055571623373f,RE-A447,NaN,RE-A447,NaN,NaN,Rosswag GmbH,Ni-1486-L75,NaN,01531,...,NaN,NaN,9df74ac3da55b295a7425df104bc6d50,RE-A447,NaN,K003,NaN,0.0,NaN,as-built


In [ ]:
# Create comprehensive statistics summary and save to CSV files
import pandas as pd
from datetime import datetime

def create_dataset_summary():
    """Create a comprehensive summary of all datasets and statistics"""
    
    # 1. Basic Dataset Information
    dataset_info = []
    
    # Main datasets info
    datasets = {
        'specimens': specimens_df,
        'tensile': tensile_df, 
        'polish': polish_df,
        'notch': notch_df
    }
    
    for name, df in datasets.items():
        dataset_info.append({
            'dataset': name,
            'total_records': len(df),
            'columns_count': len(df.columns),
            'columns_list': ', '.join(df.columns.tolist())
        })
    
    dataset_summary_df = pd.DataFrame(dataset_info)
    
    # 2. Linked Dataset Statistics
    linked_stats = []
    linked_stats.append({
        'metric': 'Complete dataset shape (rows)',
        'value': complete_dataset.shape[0]
    })
    linked_stats.append({
        'metric': 'Complete dataset shape (columns)', 
        'value': complete_dataset.shape[1]
    })
    linked_stats.append({
        'metric': 'Original specimens',
        'value': len(specimens_df)
    })
    linked_stats.append({
        'metric': 'Specimens with tensile data',
        'value': complete_dataset['name_tensile'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with polish data',
        'value': complete_dataset['name_polish'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with notch data',
        'value': complete_dataset['name_notch'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with image URLs',
        'value': len(specimens_with_images)
    })
    
    linked_stats_df = pd.DataFrame(linked_stats)
    
    # 3. Material Distribution
    material_counts = specimens_df['material'].value_counts().reset_index()
    material_counts.columns = ['material', 'count']
    material_counts['percentage'] = round((material_counts['count'] / len(specimens_df) * 100), 2)
    
    # 4. Customer Distribution (Top 15)
    customer_counts = specimens_df['customer_name'].value_counts().head(15).reset_index()
    customer_counts.columns = ['customer_name', 'count']
    customer_counts['percentage'] = round((customer_counts['count'] / len(specimens_df) * 100), 2)
    
    # 5. Image File Analysis
    images_dir = os.path.join(data_dir, 'images')
    image_analysis = []
    
    if os.path.exists(images_dir):
        image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]
        image_files_set = set(image_files)
        
        # Get database filenames
        database_filenames = set()
        for col in image_url_columns:
            if col in specimens_with_images.columns:
                non_null_values = specimens_with_images[col].dropna()
                database_filenames.update(non_null_values)
        
        matches = image_files_set.intersection(database_filenames)
        files_not_in_db = image_files_set - database_filenames
        db_not_in_files = database_filenames - image_files_set
        
        image_analysis = [
            {'metric': 'Total files in images directory', 'value': len(image_files)},
            {'metric': 'Unique filenames in database', 'value': len(database_filenames)},
            {'metric': 'Matching files (DB and directory)', 'value': len(matches)},
            {'metric': 'Files in directory but not in DB', 'value': len(files_not_in_db)},
            {'metric': 'DB entries not found in directory', 'value': len(db_not_in_files)}
        ]
    else:
        image_analysis = [{'metric': 'Images directory', 'value': 'NOT FOUND'}]
    
    image_analysis_df = pd.DataFrame(image_analysis)
    
    # 6. Analysis Views Summary
    analysis_summary = []
    for view_name, view_df in analysis_views.items():
        analysis_summary.append({
            'view_name': view_name,
            'specimen_count': len(view_df),
            'percentage_of_total': round((len(view_df) / len(specimens_df) * 100), 2)
        })
    
    analysis_views_df = pd.DataFrame(analysis_summary)
    
    # 7. Tensile Test Statistics (if data exists)
    tensile_stats_df = pd.DataFrame()
    if not tensile_df.empty:
        numeric_cols = tensile_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            tensile_stats = tensile_df[numeric_cols].describe().round(3)
            tensile_stats_df = tensile_stats.reset_index()
    
    # 8. Polish Test Statistics (if data exists) 
    polish_stats_df = pd.DataFrame()
    polish_with_data = polish_df[polish_df['porosity'].notna()]
    if not polish_with_data.empty:
        polish_summary = []
        polish_summary.append({
            'metric': 'Specimens with porosity data',
            'value': len(polish_with_data)
        })
        polish_summary.append({
            'metric': 'Porosity minimum',
            'value': round(polish_with_data['porosity'].min(), 2)
        })
        polish_summary.append({
            'metric': 'Porosity maximum', 
            'value': round(polish_with_data['porosity'].max(), 2)
        })
        polish_summary.append({
            'metric': 'Porosity average',
            'value': round(polish_with_data['porosity'].mean(), 2)
        })
        polish_stats_df = pd.DataFrame(polish_summary)
    
    return {
        'dataset_summary': dataset_summary_df,
        'linked_statistics': linked_stats_df,
        'material_distribution': material_counts,
        'customer_distribution': customer_counts,
        'image_analysis': image_analysis_df,
        'analysis_views': analysis_views_df,
        'tensile_statistics': tensile_stats_df,
        'polish_statistics': polish_stats_df
    }

# Generate all summaries
summaries = create_dataset_summary()

# Save each summary to a separate CSV file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for summary_name, summary_df in summaries.items():
    if not summary_df.empty:
        filename = f"summary_{summary_name}_{timestamp}.csv"
        summary_df.to_csv(filename, index=False)
        print(f"Saved {summary_name} to {filename}")

# Create a master summary file with key statistics
master_summary = []
master_summary.extend([
    ['Dataset', 'Metric', 'Value'],
    ['General', 'Total specimens in database', len(specimens_df)],
    ['General', 'Complete dataset columns', complete_dataset.shape[1]],
    ['General', 'Specimens with tensile data', complete_dataset['name_tensile'].notna().sum()],
    ['General', 'Specimens with polish data', complete_dataset['name_polish'].notna().sum()],
    ['General', 'Specimens with notch data', complete_dataset['name_notch'].notna().sum()],
    ['General', 'Specimens with images', len(specimens_with_images)],
    ['Materials', 'Total unique materials', specimens_df['material'].nunique()],
    ['Materials', 'Most common material', specimens_df['material'].mode()[0] if not specimens_df['material'].empty else 'N/A'],
    ['Customers', 'Total unique customers', specimens_df['customer_name'].nunique()],
    ['Images', 'Image directory exists', 'Yes' if os.path.exists(os.path.join(data_dir, 'images')) else 'No']
])

# Add polish statistics if available
if not polish_df[polish_df['porosity'].notna()].empty:
    polish_data = polish_df[polish_df['porosity'].notna()]
    master_summary.extend([
        ['Polish Tests', 'Specimens with porosity data', len(polish_data)],
        ['Polish Tests', 'Average porosity', round(polish_data['porosity'].mean(), 3)]
    ])

master_summary_df = pd.DataFrame(master_summary[1:], columns=master_summary[0])
master_filename = f"master_summary_{timestamp}.csv"
master_summary_df.to_csv(master_filename, index=False)

print(f"\n=== SUMMARY FILES CREATED ===")
print(f"Master summary saved to: {master_filename}")
print("\nIndividual summary files:")
for summary_name in summaries.keys():
    if not summaries[summary_name].empty:
        print(f"  - summary_{summary_name}_{timestamp}.csv")

# Display master summary
print(f"\n=== MASTER SUMMARY ===")
print(master_summary_df.to_string(index=False))

Saved dataset_summary to summary_dataset_summary_20251204_081042.csv
Saved linked_statistics to summary_linked_statistics_20251204_081042.csv
Saved material_distribution to summary_material_distribution_20251204_081042.csv
Saved customer_distribution to summary_customer_distribution_20251204_081042.csv
Saved image_analysis to summary_image_analysis_20251204_081042.csv
Saved analysis_views to summary_analysis_views_20251204_081042.csv
Saved tensile_statistics to summary_tensile_statistics_20251204_081042.csv
Saved polish_statistics to summary_polish_statistics_20251204_081042.csv

=== SUMMARY FILES CREATED ===
Master summary saved to: master_summary_20251204_081042.csv

Individual summary files:
  - summary_dataset_summary_20251204_081042.csv
  - summary_linked_statistics_20251204_081042.csv
  - summary_material_distribution_20251204_081042.csv
  - summary_customer_distribution_20251204_081042.csv
  - summary_image_analysis_20251204_081042.csv
  - summary_analysis_views_20251204_081042.

In [ ]:
# Create comprehensive statistics summary and save to a single text file
import pandas as pd
from datetime import datetime
import pandas as pd
import numpy as np
import os
# Read the CSV files

data_dir = '..\data\specimens_export_20251028_192412'
specimens_path = os.path.join(data_dir, 'specimens.csv')
tensile_path = os.path.join(data_dir, 'tensile_specimens.csv')
polish_path = os.path.join(data_dir, 'polish_specimens.csv')
notch_path = os.path.join(data_dir, 'notch_specimens.csv')

specimens_df = pd.read_csv(specimens_path, sep=';')
tensile_df = pd.read_csv(tensile_path, sep=';')
polish_df = pd.read_csv(polish_path, sep=';')
notch_df = pd.read_csv(notch_path, sep=';')

# Display basic information about each dataframe
print("=== SPECIMENS (Main) ===")
print(f"Shape: {specimens_df.shape}")
print(f"Columns: {list(specimens_df.columns)}")
print("\n=== TENSILE SPECIMENS ===")
print(f"Shape: {tensile_df.shape}")
print(f"Columns: {list(tensile_df.columns)}")
print("\n=== POLISH SPECIMENS ===")
print(f"Shape: {polish_df.shape}")
print(f"Columns: {list(polish_df.columns)}")
print("\n=== NOTCH SPECIMENS ===")
print(f"Shape: {notch_df.shape}")
print(f"Columns: {list(notch_df.columns)}")



def create_dataset_summary():
    """Create a comprehensive summary of all datasets and statistics"""
    
    # 1. Basic Dataset Information
    dataset_info = []
    
    # Main datasets info
    datasets = {
        'specimens': specimens_df,
        'tensile': tensile_df, 
        'polish': polish_df,
        'notch': notch_df
    }
    
    for name, df in datasets.items():
        dataset_info.append({
            'dataset': name,
            'total_records': len(df),
            'columns_count': len(df.columns),
            'columns_list': ', '.join(df.columns.tolist())
        })
    
    dataset_summary_df = pd.DataFrame(dataset_info)
    
    # 2. Linked Dataset Statistics
    linked_stats = []
    linked_stats.append({
        'metric': 'Complete dataset shape (rows)',
        'value': complete_dataset.shape[0]
    })
    linked_stats.append({
        'metric': 'Complete dataset shape (columns)', 
        'value': complete_dataset.shape[1]
    })
    linked_stats.append({
        'metric': 'Original specimens',
        'value': len(specimens_df)
    })
    linked_stats.append({
        'metric': 'Specimens with tensile data',
        'value': complete_dataset['name_tensile'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with polish data',
        'value': complete_dataset['name_polish'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with notch data',
        'value': complete_dataset['name_notch'].notna().sum()
    })
    linked_stats.append({
        'metric': 'Specimens with image URLs',
        'value': len(specimens_with_images)
    })
    
    linked_stats_df = pd.DataFrame(linked_stats)
    
    # 3. Material Distribution
    material_counts = specimens_df['material'].value_counts().reset_index()
    material_counts.columns = ['material', 'count']
    material_counts['percentage'] = round((material_counts['count'] / len(specimens_df) * 100), 2)
    
    # 4. Customer Distribution (Top 15)
    customer_counts = specimens_df['customer_name'].value_counts().head(15).reset_index()
    customer_counts.columns = ['customer_name', 'count']
    customer_counts['percentage'] = round((customer_counts['count'] / len(specimens_df) * 100), 2)
    
    # 5. Image File Analysis
    images_dir = os.path.join(data_dir, 'images')
    image_analysis = []
    
    if os.path.exists(images_dir):
        image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]
        image_files_set = set(image_files)
        
        # Get database filenames
        database_filenames = set()
        for col in image_url_columns:
            if col in specimens_with_images.columns:
                non_null_values = specimens_with_images[col].dropna()
                database_filenames.update(non_null_values)
        
        matches = image_files_set.intersection(database_filenames)
        files_not_in_db = image_files_set - database_filenames
        db_not_in_files = database_filenames - image_files_set
        
        image_analysis = [
            {'metric': 'Total files in images directory', 'value': len(image_files)},
            {'metric': 'Unique filenames in database', 'value': len(database_filenames)},
            {'metric': 'Matching files (DB and directory)', 'value': len(matches)},
            {'metric': 'Files in directory but not in DB', 'value': len(files_not_in_db)},
            {'metric': 'DB entries not found in directory', 'value': len(db_not_in_files)}
        ]
    else:
        image_analysis = [{'metric': 'Images directory', 'value': 'NOT FOUND'}]
    
    image_analysis_df = pd.DataFrame(image_analysis)
    
    # 6. Analysis Views Summary
    analysis_summary = []
    for view_name, view_df in analysis_views.items():
        analysis_summary.append({
            'view_name': view_name,
            'specimen_count': len(view_df),
            'percentage_of_total': round((len(view_df) / len(specimens_df) * 100), 2)
        })
    
    analysis_views_df = pd.DataFrame(analysis_summary)
    
    # 7. Tensile Test Statistics (if data exists)
    tensile_stats_df = pd.DataFrame()
    if not tensile_df.empty:
        numeric_cols = tensile_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            tensile_stats = tensile_df[numeric_cols].describe().round(3)
            tensile_stats_df = tensile_stats.reset_index()
    
    # 8. Polish Test Statistics (if data exists) 
    polish_stats_df = pd.DataFrame()
    polish_with_data = polish_df[polish_df['porosity'].notna()]
    if not polish_with_data.empty:
        polish_summary = []
        polish_summary.append({
            'metric': 'Specimens with porosity data',
            'value': len(polish_with_data)
        })
        polish_summary.append({
            'metric': 'Porosity minimum',
            'value': round(polish_with_data['porosity'].min(), 2)
        })
        polish_summary.append({
            'metric': 'Porosity maximum', 
            'value': round(polish_with_data['porosity'].max(), 2)
        })
        polish_summary.append({
            'metric': 'Porosity average',
            'value': round(polish_with_data['porosity'].mean(), 2)
        })
        polish_stats_df = pd.DataFrame(polish_summary)
    
    return {
        'dataset_summary': dataset_summary_df,
        'linked_statistics': linked_stats_df,
        'material_distribution': material_counts,
        'customer_distribution': customer_counts,
        'image_analysis': image_analysis_df,
        'analysis_views': analysis_views_df,
        'tensile_statistics': tensile_stats_df,
        'polish_statistics': polish_stats_df
    }

# Generate all summaries
summaries = create_dataset_summary()

# Create timestamp for filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_filename = f"comprehensive_dataset_report_{timestamp}.txt"

# Write all summaries to a single text file
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("COMPREHENSIVE DATASET ANALYSIS REPORT\n")
    f.write("=" * 80 + "\n")
    f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    # 1. Dataset Summary
    f.write("1. BASIC DATASET INFORMATION\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['dataset_summary'].to_string(index=False) + "\n\n")
    
    # 2. Linked Statistics
    f.write("2. LINKED DATASET STATISTICS\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['linked_statistics'].to_string(index=False) + "\n\n")
    
    # 3. Material Distribution
    f.write("3. MATERIAL DISTRIBUTION\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['material_distribution'].to_string(index=False) + "\n\n")
    
    # 4. Customer Distribution
    f.write("4. CUSTOMER DISTRIBUTION (TOP 15)\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['customer_distribution'].to_string(index=False) + "\n\n")
    
    # 5. Image Analysis
    f.write("5. IMAGE FILE ANALYSIS\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['image_analysis'].to_string(index=False) + "\n\n")
    
    # 6. Analysis Views
    f.write("6. ANALYSIS VIEWS SUMMARY\n")
    f.write("-" * 40 + "\n")
    f.write(summaries['analysis_views'].to_string(index=False) + "\n\n")
    
    # 7. Tensile Statistics
    if not summaries['tensile_statistics'].empty:
        f.write("7. TENSILE TEST STATISTICS\n")
        f.write("-" * 40 + "\n")
        f.write(summaries['tensile_statistics'].to_string(index=False) + "\n\n")
    
    # 8. Polish Statistics
    if not summaries['polish_statistics'].empty:
        f.write("8. POLISH TEST STATISTICS\n")
        f.write("-" * 40 + "\n")
        f.write(summaries['polish_statistics'].to_string(index=False) + "\n\n")
    
    # Summary section
    f.write("=" * 80 + "\n")
    f.write("EXECUTIVE SUMMARY\n")
    f.write("=" * 80 + "\n")
    f.write(f"Total specimens in database: {len(specimens_df)}\n")
    f.write(f"Complete dataset columns: {complete_dataset.shape[1]}\n")
    f.write(f"Specimens with tensile data: {complete_dataset['name_tensile'].notna().sum()}\n")
    f.write(f"Specimens with polish data: {complete_dataset['name_polish'].notna().sum()}\n")
    f.write(f"Specimens with notch data: {complete_dataset['name_notch'].notna().sum()}\n")
    f.write(f"Specimens with images: {len(specimens_with_images)}\n")
    f.write(f"Total unique materials: {specimens_df['material'].nunique()}\n")
    f.write(f"Most common material: {specimens_df['material'].mode()[0] if not specimens_df['material'].empty else 'N/A'}\n")
    f.write(f"Total unique customers: {specimens_df['customer_name'].nunique()}\n")
    f.write(f"Image directory exists: {'Yes' if os.path.exists(os.path.join(data_dir, 'images')) else 'No'}\n")
    
    if not polish_df[polish_df['porosity'].notna()].empty:
        polish_data = polish_df[polish_df['porosity'].notna()]
        f.write(f"Specimens with porosity data: {len(polish_data)}\n")
        f.write(f"Average porosity: {round(polish_data['porosity'].mean(), 3)}\n")

print(f"Comprehensive dataset report saved to: {report_filename}")
print(f"Report size: {os.path.getsize(report_filename)} bytes")

=== SPECIMENS (Main) ===
Shape: (2437, 26)
Columns: ['key', 'name', 'description', 'number', 'sub_sales_order_key', 'sub_sales_order_number', 'customer_name', 'material', 'build_job_key', 'build_job_number', 'print_sub_sales_order_key', 'machine', 'powder_id', 'lab_run_number', 'related_files', 'comment', 'polish_comment', 'tensile_comment', 'notch_comment', 'chemical_comment', 'rigid_comment', 'fatigue_comment', 'external_comment', 'flag_import', 'no_specimens_detail', 'no_specimens_total']

=== TENSILE SPECIMENS ===
Shape: (1623, 16)
Columns: ['key', 'name', 'description', 'number', 'specimens_key', 'norm', 'shape_of_specimens', 'orientation', 'temperature', 'yield_limit', 'yield_strength', 'tensile_strength', 'elongation_at_break', 'quadrant_build_in', 'heat_treatment', 'contraction_at_break']

=== POLISH SPECIMENS ===
Shape: (7093, 66)
Columns: ['key', 'name', 'description', 'number', 'specimens_key', 'density', 'volume_energy_density', 'quadrant_build_in', 'heat_treatment', 'polis

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
C:\Users\perrydebussy\AppData\Local\Temp\ipykernel_30732\3506163166.py:9: SyntaxWarning: invalid escape sequence '\d'
  data_dir = '..\data\specimens_export_20251028_192412'


```
polishSpecimensDescr = dict(
    key=str(), # the key
    name=str(), # Firmenname ?? what is this column here ??
    description=str(), # description of specimens
    number=str(), # Probennummer
    specimens_key=str(), # Link zur Hauptprobennummer
    density=None, # Dichte in %
    volume_energy_density=float(), # Volumenenergiedichte in W/mm^3# redundant as it will be in sample data in AddiBase
    quadrant_build_in=str(), # Angabe in welchem Quadrant die Probe gedruckt wurde
    heat_treatment=str(), # Wärmebehandlung
    polish_img_url_1=str(), # url_thumb; url_orig Porosität
    polish_img_url_2=str(), # url_thumb; url_orig Porosität
    polish_img_url_3=str(), # url_thumb; url_orig Porosität
    polish_img_url_4=str(), # url_thumb; url_orig Gefüge
    polish_img_url_5=str(), # url_thumb; url_orig Gefüge
    polish_img_url_6=str(), # url_thumb; url_orig Gefüge
    pixel_per_mm_1=None, # ppm Porosität (not used any more in form)
    pixel_per_mm_2=None, # ppm Porosität (not used any more in form)
    pixel_per_mm_3=None, # ppm Porosität (not used any more in form)
    pixel_per_mm_4=None, # ppm Gefüge (not used any more in form)
    pixel_per_mm_5=None, # ppm Gefüge (not used any more in form)
    pixel_per_mm_6=None, # ppm Gefüge (not used any more in form)
    porosity_1=None, # lab result
    porosity_2=None, # lab result
    porosity_3=None, # lab result
    porosity_algorithm=None, # used for addibase result
    density_algorithm=None,  # used for addibase result
    grain_size_algorithm=None, # used for addibase result
    polish_plane=str(),     # example usage: drop-down X-Y -> only first image
    etching=str(),          # Nital 2% for 10 seconds: Drop Down dependent on Material
    microstructure=list(),  # e.g. Martensitic with retained austenite: Drop Down depending on Material
    grain_size=None,        # in mu (e.g. 20)
    porosity=None,          # in %, e.g. 3,5
    has_images=False,       # an indicator if images are here
    number_of_images=int(), # counts the images to show in table
    grain_size_4=None, # lab result Korngröße
    grain_size_5=None, # lab result Korngröße
    grain_size_6=None, # lab result Korngröße
    zoom_factor_1=None, # zoom Porosität
    zoom_factor_2=None, # zoom Porosität
    zoom_factor_3=None, # zoom Porosität
    zoom_factor_4=None, # zoom Gefüge
    zoom_factor_5=None, # zoom Gefüge
    zoom_factor_6=None, # zoom Gefüge
    polish_img_url_3a=str(), # url_thumb; url_orig 4th image of Porosität
    polish_img_url_6a=str(), # url_thumb; url_orig 4th image of Gefüge
    polish_img_url_1_orig=str(), # not used any more
    polish_img_url_2_orig=str(), # not used any more
    polish_img_url_3_orig=str(), # not used any more
    polish_img_url_4_orig=str(), # not used any more
    polish_img_url_5_orig=str(), # not used any more
    polish_img_url_6_orig=str(), # not used any more
    polish_img_url_3a_orig=str(), # not used any more
    polish_img_url_6a_orig=str(), # not used any more
    polish_plane_2=str(), # polish plane Porosität
    polish_plane_3=str(), # polish plane Porosität
    polish_plane_3a=str(), # polish plane Porosität
    polish_plane_4=str(), # polish plane Gefüge
    polish_plane_5=str(), # polish plane Gefüge
    polish_plane_6=str(), # polish plane Gefüge
    polish_plane_6a=str(), # polish plane Gefüge
    zoom_factor_3a=None, # zoom Porosität
    zoom_factor_6a=None, # zoom Gefüge
    pixel_per_mm_3a=None, # ppm Porosität (not used any more in form)
    pixel_per_mm_6a=None, # ppm Gefüge (not used any more in form)
    porosity_3a=None, # lab result Porosität
    grain_size_6a=None, # lab result Korngröße
)
```